In [4]:
#MySQL 연결
import pymysql
from sqlalchemy import create_engine, text #연결 관리용으로만 가볍게
from pathlib import Path
from pprint import pprint

#데이터 처리/시각화
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

#외부 API 호출하기
import requests
from requests.exceptions import Timeout, ConnectionError, RequestException
from google.cloud import bigquery

#.env 파일 활용하기
from dotenv import load_dotenv
import os
import time

#env 파일 읽기
load_dotenv()

#API 요청 정보
url = 'https://bigdata.kepco.co.kr/openapi/v1/powerUsage/houseAve.do'
api_key = os.getenv('ELECTRIC_API_KEY')

#db에서 level 1인 지역 코드를 가져왔다. 앞의 두자리를 잘라서
#사용할 예정입니다.
region = [
"1100000000","2600000000","2700000000","2800000000","2900000000","3000000000","3100000000","3600000000","4100000000","4300000000","4400000000","4600000000","4700000000","4800000000","5000000000","5100000000","5200000000"
]

#SQLAlchemy의 engine을 미리 만들어 놓습니다.
DB_URL = f"mysql+pymysql://{os.getenv('DB_USERNAME')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
my_engine = create_engine(DB_URL)

In [5]:
#구글 bigquery 연결을 만듭니다.
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'gcp-credentials.json'

project_id = os.getenv('GCP_PROJECT_ID')
dataset = os.getenv('BQ_DATASET')
big_engine = create_engine(f'bigquery://{project_id}/{dataset}')

In [6]:
query = text("""
    select *
    from household_power
""")

with my_engine.connect() as conn:
    result = conn.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

In [7]:
df.head()

,id,sd_code,year,month,sd_name,sgg_name,house_cnt,power_usage,bill
0,1,11,2013,5,서울특별시,중구,61652,211.55,26835
1,2,11,2013,5,서울특별시,용산구,115636,227.71,32179
2,3,11,2013,5,서울특별시,광진구,167472,200.41,24371
3,4,11,2013,5,서울특별시,중랑구,172540,213.22,24754
4,5,11,2013,5,서울특별시,양천구,177707,243.22,29801


In [8]:
# === 1단계: 부천시 데이터 추출 ===
부천시 = df[df['sgg_name'].str.contains('부천시')]

# === 2단계: 가중평균을 포함한 집계 ===
부천시_합계 = 부천시.groupby(['year', 'month']).apply(
    lambda x: pd.Series({
        'sd_code': x['sd_code'].iloc[0],          # 그룹 내 첫 번째 값 (어차피 다 41)
        'sd_name': x['sd_name'].iloc[0],           # 그룹 내 첫 번째 값 (어차피 다 경기도)
        'sgg_name': '부천시',                       # 새 이름 지정
        'house_cnt': x['house_cnt'].sum(),          # 가구수는 단순 합계
        'power_usage': (x['power_usage'] * x['house_cnt']).sum() / x['house_cnt'].sum(),
            # 각 구의 (평균 사용량 × 가구수)를 합한 뒤, 총 가구수로 나눔 = 가중평균
        'bill': (x['bill'] * x['house_cnt']).sum() / x['house_cnt'].sum(),
            # 전기요금도 동일한 가중평균
    })
).reset_index()
# reset_index(): groupby 결과에서 year, month가 인덱스로 들어가 있는데,
#                이걸 다시 일반 컬럼으로 되돌림

부천시_합계['power_usage'] = 부천시_합계['power_usage'].round(2)
부천시_합계['bill'] = 부천시_합계['bill'].round(0).astype(int)

max_id = df['id'].max()
부천시_합계['id'] = range(max_id + 1, max_id + 1 + len(부천시_합계))

# === 3단계: 기존 df에서 부천시 구 단위 데이터 제거 ===
df = df[~df['sgg_name'].str.contains('부천시')]
# ~는 NOT 연산자. 부천시가 포함되지 않은 행만 남김

# === 4단계: 합친 부천시 데이터를 df에 붙이기 ===
df = pd.concat([df, 부천시_합계], ignore_index=True)
# ignore_index=True: 인덱스를 0부터 새로 매김 (안 하면 인덱스 중복 발생)

# === 확인 ===
df[df['sgg_name'] == '부천시'].head()

,id,sd_code,year,month,sd_name,sgg_name,house_cnt,power_usage,bill
37835,38124,41,2013,5,경기도,부천시,119964,286.72,34598
37836,38125,41,2013,6,경기도,부천시,119958,297.97,37187
37837,38126,41,2013,7,경기도,부천시,119961,315.47,42033
37838,38127,41,2013,8,경기도,부천시,119982,731.05,260935
37839,38128,41,2013,9,경기도,부천시,120025,-38.89,-157448


In [9]:
#전력사용량, 전기요금 평균을 숫자 타입으로 변경합니다.
df['power_usage'] = pd.to_numeric(df['power_usage'], errors='coerce')
df['bill'] = pd.to_numeric(df['bill'], errors='coerce')

for col in ['power_usage', 'bill']:
    # 음수를 NaN으로
    df[col] = df[col].where(df[col] >= 0)

# 지역별로 정렬 후 그룹 내에서 보간
df = df.sort_values(['sgg_name', 'year', 'month'])
df[['power_usage', 'bill']] = (
    df.groupby(['sd_name','sgg_name'])[['power_usage', 'bill']]
    .transform(lambda x: x.interpolate())
)

In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 37987 entries, 74 to 37813
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           37987 non-null  int64  
 1   sd_code      37987 non-null  int64  
 2   year         37987 non-null  int64  
 3   month        37987 non-null  int64  
 4   sd_name      37987 non-null  str    
 5   sgg_name     37987 non-null  str    
 6   house_cnt    37987 non-null  int64  
 7   power_usage  37987 non-null  float64
 8   bill         37987 non-null  float64
dtypes: float64(2), int64(5), str(2)
memory usage: 3.7 MB


In [11]:
client = bigquery.Client()
table_id = f'{project_id}.{dataset}.raw'

job_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE'  # 기존 데이터 날리고 새로 넣기
)

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()
print(f'{job.output_rows}행 입력 완료')

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


37987행 입력 완료
